<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_4/lessons/lesson_43_llm_api/note_lesson_43_llm_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Урок 43 — Інтеграція LLM API: новинний агрегатор, крок 7

| Крок | Що робимо |
|---|---|
| 0 | `news_hub` з аналізом LLM; pytest з ноутбука |
| 1 | один інтерфейс: `FakeLLM` (вправа 1) |
| 2 | відповідь — JSON за схемою, перевірка Pydantic і повтор (вправи 2–3) |
| 3 | промпт-ін'єкція (вправа 4) |
| 4 | пул моделей Gemini: коли переходити, коли зупинитись (вправи 5–6) |
| 5 | circuit breaker і атомарний лічильник (вправи 7–8) |
| 6 | API: база → кеш → модель (вправа 9) |
| 7 | знайди помилку: схема, яку не приймає Gemini (вправа 10) |
| ★ | необов'язково: справжній виклик Gemini з ключем |

Ключ **не потрібен**: модель підміняє `FakeLLM`, база — SQLite, Redis — `fakeredis`. Теорія — на [сторінці уроку](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_43/).

---
## 0. Проєкт і pytest

Клітинка завантажує `news_hub` (у Colab), ставить залежності й переходить у папку проєкту. `run_pytest(...)` запускає pytest окремим процесом і повертає вивід. `await` можна писати прямо в клітинці.

In [ ]:
import os
import subprocess
import sys

LESSON = "module_4/lessons/lesson_43_llm_api"
REPO = "https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026.git"

if not os.path.isdir("news_hub"):
    if not os.path.isdir("course_repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "course_repo"], check=True)
        subprocess.run(["git", "-C", "course_repo", "sparse-checkout", "set", LESSON], check=True)
    os.chdir(os.path.join("course_repo", LESSON))

try:
    import aiosqlite, anthropic, fakeredis, fastapi, httpx, pytest_asyncio  # noqa: F401
    from google import genai  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "news_hub/requirements.txt"], check=True)

os.chdir("news_hub")
sys.path.insert(0, os.getcwd())
os.environ["DATABASE_URL"] = "sqlite+aiosqlite:///lesson43.db"
os.environ["REDIS_URL"] = "fakeredis://"
os.environ["LLM_PROVIDER"] = "fake"          # застосунок у вправі 9 — без ключа й мережі


def run_pytest(*args: str) -> str:
    """pytest окремим процесом (як у терміналі); повертає вивід."""
    env = {key: value for key, value in os.environ.items() if key not in ("DATABASE_URL", "REDIS_URL", "LLM_PROVIDER")}
    result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no", *args],
                            capture_output=True, text=True, env=env)
    return result.stdout + result.stderr


print(run_pytest().splitlines()[-1])

---
## 1. Один інтерфейс

Будь-який провайдер — `GeminiClient`, `AnthropicClient`, `FakeLLM` — має метод `await generate(prompt, system=..., schema=...)` і повертає `LLMReply(text, model, input_tokens, output_tokens)`. `FakeLLM(replies)` відповідає за сценарієм: рядки — як відповідь моделі, винятки — кидає; `fake.calls` — усі промпти, які він отримав.

### Вправа 1

Створи `fake` зі сценарієм: спершу відповідь `ANSWER`, потім виняток `LLMUnavailable("503")`. Виклич `generate` двічі: першу відповідь збережи в `reply`, другий виклик має кинути виняток — збережи його в `error`.

In [ ]:
import json

from news_hub.analysis import SYSTEM_PROMPT, NewsAnalysis, analyze_news, build_prompt
from news_hub.llm import FakeLLM, LLMUnavailable

ANSWER = json.dumps({"summary": "НБУ знизив облікову ставку до 13%.", "category": "Економіка",
                     "sentiment": "позитивна", "keywords": ["НБУ", "облікова ставка"]}, ensure_ascii=False)

# YOUR CODE HERE

print(reply)
print("виняток:", repr(error), "| промптів отримано:", len(fake.calls))
assert reply.text == ANSWER and reply.model == "fake"
assert isinstance(error, LLMUnavailable) and fake.calls == ["перший промпт", "другий промпт"]
print("✅ Вправа 1 пройдена")

---
## 2. Відповідь — JSON за схемою

`NewsAnalysis` — і схема для моделі, і перевірка відповіді: `summary` 5–300 символів, `category` — з переліку тем, `sentiment` — одна з трьох, `keywords` — не більше 5, інших полів не буває.

In [ ]:
print(json.dumps(NewsAnalysis.model_json_schema(), ensure_ascii=False, indent=1)[:900])

### Вправа 2

Перевір кожну відповідь з `answers` моделлю `NewsAnalysis.model_validate_json`. У `verdicts` збережи для кожної `"ok"` або назву **першого** поля з помилкою (`error.errors()[0]["loc"][0]`; для не-JSON — `"json"`).

In [ ]:
from pydantic import ValidationError


def variant(**changes: object) -> str:
    return json.dumps(json.loads(ANSWER) | changes, ensure_ascii=False)


answers = [
    ANSWER,
    variant(sentiment="дуже погана"),
    variant(category="Новини"),
    variant(keywords=["a", "b", "c", "d", "e", "f"]),
    'Звісно! Ось аналіз: {"summary": ...}',
    variant(mood="радісний"),
]

# YOUR CODE HERE

for text, verdict in zip(answers, verdicts):
    print(f"{verdict:10} ← {text[:70]}")
assert verdicts == ["ok", "sentiment", "category", "keywords", "json", "mood"]
print("✅ Вправа 2 пройдена")

**🔮 Прогноз:** Схему вже передано моделі в запиті (`response_json_schema`). Навіщо тоді перевіряти відповідь у коді?

<details>
<summary>Відповідь</summary>

Схема зменшує ймовірність браку, але не гарантує: відповідь може обрізатись на `max_output_tokens`, бути порожньою (заблокована), а обмеження на кшталт довжини провайдер може не застосувати. Перевірка в коді — межа, яку контролюємо ми.
</details>

### Вправа 3

`analyze_news` робить **одну** повторну спробу, якщо відповідь не пройшла перевірку. Дай йому `FakeLLM`, що спершу відповідає `answers[1]` (невалідна тональність), потім `ANSWER`. Результат — у `result`, клієнт — у `llm`. Подивись, що отримала модель у другому промпті.

In [ ]:
TITLE = "Нацбанк знизив облікову ставку до 13%"

# YOUR CODE HERE

print(result.analysis)
print("викликів:", len(llm.calls), "| токени обох спроб:", result.input_tokens)
print("кінець другого промпту:", llm.calls[1][-160:])
assert result.analysis.sentiment == "позитивна" and len(llm.calls) == 2
assert "не пройшла перевірку" in llm.calls[1]
print("✅ Вправа 3 пройдена")

---
## 3. Промпт-ін'єкція

Заголовок пише хтось інший. Якщо в ньому інструкція — модель може її виконати. Захист: інструкція — у `system`, заголовок — між `<news>` і `</news>`, і мітки з самого заголовка прибираються.

In [ ]:
print(SYSTEM_PROMPT)
print("---")
print(build_prompt("Курс долара виріс"))

### Вправа 4

Придумай заголовок `attack`, який намагається **закрити** блок даних (`</news>`) і дописати свою інструкцію після нього. Збережи промпт у `prompt`. Перевірки покажуть, чи вийшов текст атаки за межі блоку.

In [ ]:
# YOUR CODE HERE

print(prompt)
inside = prompt.split("<news>")[1].split("</news>")[0]
assert "</news>" in attack or "<news>" in attack, "атака має містити мітку"
assert prompt.count("<news>") == 1 and prompt.count("</news>") == 1
assert "Ігноруй" in inside or len(inside.strip()) > 0
print("✅ Вправа 4 пройдена — текст атаки лишився всередині даних")

**🔮 Прогноз:** Мітки захищають не на 100% — модель однаково «бачить» текст «Ігноруй інструкції». Що найгірше може статися в `news_hub`, якщо вона послухається?

<details>
<summary>Відповідь</summary>

Хибна тональність — одна з трьох дозволених, або зайве ключове слово. Не більше: схема не дає нових полів чи довільного тексту, а `news_hub` не виконує відповідь моделі (не запускає код, не йде за посиланнями, не складає SQL рядком). Шкоду обмежує те, що застосунок **робить** з відповіддю.
</details>

---
## 4. Пул моделей Gemini

`GeminiClient` пробує моделі з `MODEL_POOL` по черзі. Рішення — за кодом помилки: 404 / 429 / 5xx / тайм-аут → наступна модель; 400 / 401 / 403 — помилка **нашого** запиту, інша модель не допоможе.

`gemini_with(*відповіді)` з `tests/unit/test_llm.py` — справжній `GeminiClient`, у якого HTTP-виклик підмінено: кожна «відповідь» — або `gemini_error(код)`, або `gemini_ok()`. Мережі немає.

In [ ]:
from news_hub.llm import MODEL_POOL, RETRY_NEXT_MODEL
from tests.unit.test_llm import gemini_error, gemini_ok, gemini_with

print("пул:", MODEL_POOL)
print("наступна модель при:", sorted(RETRY_NEXT_MODEL))

### Вправа 5

Перша модель перевантажена (503), у другої вичерпана квота (429), третя відповідає. Створи `client, fake_models` через `gemini_with(...)`, виклич `generate` і збережи відповідь у `reply5`. Яка модель відповіла?

In [ ]:
# YOUR CODE HERE

print("спроби:", [call["model"] for call in fake_models.calls])
print("відповіла:", reply5.model, "| токени:", reply5.input_tokens, "+", reply5.output_tokens)
assert reply5.model == MODEL_POOL[2] and len(fake_models.calls) == 3
print("✅ Вправа 5 пройдена")

### Вправа 6

Ключ невалідний: Gemini відповідає `400` («API key not valid»). Переконайся, що клієнт **не** перебирає пул: збережи виняток у `error6`, клієнт з фейком — у `client6, models6`.

In [ ]:
# YOUR CODE HERE

print(repr(error6))
print("спроб:", len(models6.calls))
assert isinstance(error6, LLMUnavailable) and len(models6.calls) == 1
print("✅ Вправа 6 пройдена")

---
## 5. Circuit breaker

`GuardedLLM(клієнт, CircuitBreaker(redis, threshold=N))` — той самий інтерфейс. Після `N` збоїв поспіль (`LLMUnavailable`) breaker «відкривається»: наступні виклики одразу кидають `CircuitOpen(retry_after)`, провайдера не чіпаючи. Ключ `cb:llm:open` з TTL — сам зникне через `cooldown` секунд.

### Вправа 7

Провайдер лежить (кожна відповідь — `LLMUnavailable`). Зроби 5 викликів через `GuardedLLM` з порогом 3 і збери назви класів винятків у `kinds`. Скільки разів провайдер справді отримав запит?

In [ ]:
import fakeredis

from news_hub.llm import CircuitBreaker, CircuitOpen, GuardedLLM

redis = fakeredis.FakeAsyncRedis(decode_responses=True)
down = FakeLLM([LLMUnavailable("503")] * 5)

# YOUR CODE HERE

print(kinds)
print("провайдер отримав запитів:", len(down.calls), "| TTL cb:llm:open:", await redis.ttl("cb:llm:open"))
assert kinds == ["LLMUnavailable"] * 3 + ["CircuitOpen"] * 2 and len(down.calls) == 3
print("✅ Вправа 7 пройдена")

### Вправа 8. Лічильник збоїв: `GET` + `SET` проти `INCR`

`LaggyRedis` — `fakeredis` із затримкою мережі перед кожною командою (як справжній Redis: між запитом і відповіддю — мережа). 20 збоїв стаються **одночасно**.

1. Запусти `naive_failure` 20 разів через `asyncio.gather` — скільки збоїв порахував лічильник?
2. Напиши `atomic_failure`, що рахує збій **однією** командою Redis, і перевір, що лічильник — 20.

In [ ]:
import asyncio


class LaggyRedis:
    """Redis із затримкою мережі: між командами інші корутини встигають виконатись."""

    def __init__(self) -> None:
        self._redis = fakeredis.FakeAsyncRedis(decode_responses=True)

    async def get(self, key: str) -> str | None:
        await asyncio.sleep(0.001)
        return await self._redis.get(key)

    async def set(self, key: str, value: object, ex: int | None = None) -> None:
        await asyncio.sleep(0.001)
        await self._redis.set(key, value, ex=ex)

    async def incr(self, key: str) -> int:
        await asyncio.sleep(0.001)
        return await self._redis.incr(key)


laggy = LaggyRedis()


async def naive_failure() -> None:
    failures = int(await laggy.get("naive") or 0) + 1       # прочитали…
    await laggy.set("naive", failures, ex=300)              # …а поки йшла відповідь, інші теж прочитали


await asyncio.gather(*(naive_failure() for _ in range(20)))
naive_count = int(await laggy.get("naive"))

# YOUR CODE HERE

await asyncio.gather(*(atomic_failure() for _ in range(20)))
atomic_count = int(await laggy.get("atomic"))
print(f"GET + SET: {naive_count} з 20 | INCR: {atomic_count} з 20")
assert naive_count < 20 and atomic_count == 20
print("✅ Вправа 8 пройдена — з порогом 5 «наївний» breaker відкрився б набагато пізніше")

---
## 6. API: база → кеш → модель

`POST /api/news/{id}/analyze` віддає поле `source`: `db` — аналіз уже в рядку новини; `cache` — той самий заголовок уже аналізували (Redis, ключ — хеш тексту); `llm` — новий виклик моделі. Модель підміняємо через `app.dependency_overrides[get_llm_client]`.

### Вправа 9

1. Збери знімок (`POST /api/scrape`, `{"source": "snapshot"}`) і двічі проаналізуй новину `1`.
2. Видали всі новини (`DELETE /api/news`), збери знімок ще раз і знову проаналізуй новину `1`.

Збережи три значення `source` у `sources`. Скільки разів модель побачила заголовок?

In [ ]:
import httpx

from news_hub.api import app, get_llm_client
from news_hub.db import Base, engine

async with engine.begin() as conn:                            # чиста база для вправи
    await conn.run_sync(Base.metadata.drop_all)
    await conn.run_sync(Base.metadata.create_all)

model = FakeLLM()
app.dependency_overrides[get_llm_client] = lambda: model

async with app.router.lifespan_context(app):
    await app.state.redis.flushdb()
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app), base_url="http://test") as aclient:
        # YOUR CODE HERE
        pass
        stats = (await aclient.get("/api/news/stats")).json()
app.dependency_overrides.pop(get_llm_client, None)

print("sources:", sources, "| модель викликано:", len(model.calls))
print("stats:", {key: stats[key] for key in ("total", "ai_category", "sentiment")})
assert sources == ["llm", "db", "cache"] and len(model.calls) == 1
print("✅ Вправа 9 пройдена")

---
## 7. Знайди помилку: схема, яку не приймає Gemini

Перша версія `GeminiClient` передавала схему полем `response_schema=NewsAnalysis`. Усі тести з `FakeLLM` і з моком SDK були зелені. Перший запуск з мережею (ключ навмисно невалідний):

```text
LLMUnavailable: gemini відхилив запит (400): Invalid JSON payload received.
Unknown name "additional_properties" at 'generation_config.response_schema': Cannot find field.
```

### Вправа 10

1. Порівняй JSON Schema `NewsAnalysis` зі схемою тієї самої моделі **без** `extra="forbid"` (`Loose`). Збережи в `extra_keys` ключі верхнього рівня, які є лише в першій.
2. Звідки взялося `additional_properties` у запиті й чому тести цього не бачили?

In [ ]:
from pydantic import ConfigDict


class Loose(NewsAnalysis):
    __doc__ = NewsAnalysis.__doc__                        # той самий опис — різниця лише в extra
    model_config = ConfigDict(extra="ignore", str_strip_whitespace=True)


strict_schema = NewsAnalysis.model_json_schema()
loose_schema = Loose.model_json_schema()

# YOUR CODE HERE

print("лише в NewsAnalysis:", extra_keys, "→", {key: strict_schema[key] for key in extra_keys})
assert extra_keys == {"additionalProperties"}
print("✅ Вправа 10 пройдена")

<details>
<summary>Відповідь</summary>

`extra="forbid"` додає в схему `"additionalProperties": false`. Поле `response_schema` у Gemini API приймає лише підмножину схем OpenAPI — `additional_properties` там немає, і Google відхиляє **увесь** запит `400`. Тести не бачили, бо жоден не спілкувався з Google: `FakeLLM` і мок приймають будь-яку схему. Правильно — `response_json_schema=schema.model_json_schema()` (повна JSON Schema). Щоб не повторилось — контрактний тест `test_gemini_accepts_request_shape`: невалідний ключ, лише мережа — Google перевіряє структуру запиту раніше за ключ.
</details>

Контрактний тест можна запустити й тут — потрібна лише мережа (без мережі він впаде з `TimeoutError` / `ClientConnectorError`):

In [ ]:
print(run_pytest("-m", "llm", "-k", "request_shape", "-rs").splitlines()[-1])

---
## ★ Необов'язково: справжній Gemini

Потрібен ключ: [Google AI Studio → Get API key](https://aistudio.google.com/apikey). У Colab — 🔑 **Secrets** (ліва панель) → `GEMINI_API_KEY`, доступ для ноутбука. Локально — змінна середовища. Ключ **ніколи** не вписуй у клітинку: ноутбук потрапить у git разом з ним.

Клітинка аналізує 3 заголовки зі знімка й друкує, скільки токенів це коштувало. Без ключа — пропускається.

In [ ]:
from news_hub.llm import make_llm
from news_hub.snapshot import load_snapshot

key = os.getenv("GEMINI_API_KEY")
if not key:
    try:
        from google.colab import userdata          # у Colab — із Secrets
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = None

if key:
    os.environ["GEMINI_API_KEY"] = key
    gemini = make_llm("gemini")
    try:
        for raw in load_snapshot()[:3]:
            result = await analyze_news(raw["title"], gemini)
            print(f"{result.model} {result.input_tokens}+{result.output_tokens} токенів | {raw['title']}")
            print("   ", result.analysis.model_dump_json())
    finally:
        await gemini.aclose()
else:
    print("GEMINI_API_KEY немає — справжній виклик пропущено")

---
## Самоперевірка

1. Чому 400 від Gemini не веде до наступної моделі, а 429 — веде?
2. Що робить `analyze_news`, коли відповідь моделі не пройшла `NewsAnalysis`, і чому лише одна повторна спроба?
3. Які шари захищають `news_hub` від промпт-ін'єкції і який з них найважливіший?
4. Чому лічильник збоїв має бути `INCR`, і чому на звичайному `fakeredis` різниці не видно?
5. Звідки береться `source = cache` для новини, якої ще не було в базі?

## Далі

- Книга курсу: [Урок 43](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m4/lesson_43/) — покрокові діаграми пулу моделей і breaker, вартість і токени, архітектура.
- У папці `news_hub`: `LLM_PROVIDER=fake uvicorn news_hub.api:app --reload` → http://127.0.0.1:8000/docs; з ключем — `GEMINI_API_KEY=… uvicorn …`.
- Урок 44 — архітектура й патерни: `LLMClient` і `GuardedLLM` — «стратегія» і «декоратор».